### Multimodal RAG (PDF With Images)

In [6]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

<p>Read Docs :
 https://huggingface.co/docs/transformers/en/model_doc/clip

In [2]:
# CLIP Model
import os
from dotenv import load_dotenv
load_dotenv()

## set up the environment
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

# initialize the CLIP model for unified embeddings
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

e:\RAG\RAG Project KN\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\itsme\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [5]:
### Embedding Functions
# def embed_image(image: Image.Image) -> np.ndarray:
#     """Generate an embedding for an image using CLIP."""
#     inputs = clip_processor(images=image, return_tensors="pt")
#     with torch.no_grad():
#         image_features = clip_model.get_image_features(**inputs)
#     return image_features.cpu().numpy()

# def embed_text(text: str) -> np.ndarray:
#     """Generate an embedding for text using CLIP."""
#     inputs = clip_processor(text=[text], return_tensors="pt", padding=True)
#     with torch.no_grad():
#         text_features = clip_model.get_text_features(**inputs)
#     return text_features.cpu().numpy()

In [25]:
# # Embded image and text functions using the CLIP model
# def embed_image(image_data):
#     """Embed Image using CLIP """
#     if isinstance(image_data, str): #If path
#         image = Image.open(image_data).convert("RGB")
#     else:  # If PIL Image
#         image = image_data

    
#     inputs = clip_processor(images=image , return_tensors="pt")
#     with torch.no_grad():
#         features = clip_model.get_image_features(**inputs)
#         # Normalize the embeddings to unit vectors
#         features = features / features.norm(dim=-1, keepdim=True)
#         return features.squeeze().numpy()

# def embed_text(text):
#     """Embed Text using CLIP """
#     inputs = clip_processor(
#         text=text,
#         return_tensors="pt",
#         padding=True,
#         truncation=True,
#         max_length=77  # CLIP's maximum token length for text
#         )
#     with torch.no_grad():
#         # 1. Get the tuple from transformers
#         outputs = clip_model.get_text_features(**inputs, return_dict=False)
    
#         # 2. Grab the actual PyTorch tensor from index 0
#         features = outputs[0] if isinstance(outputs, tuple) else outputs
    
#         # 3. Now .norm() works on the tensor!
#         features = features / features.norm(dim=-1, keepdim=True)
#         return features.squeeze().cpu().numpy()


In [30]:
from PIL import Image
import torch

def embed_image(image_data):
    """Embed Image using CLIP"""
    if isinstance(image_data, str):  # If path
        image = Image.open(image_data).convert("RGB")
    else:  # If PIL Image
        image = image_data

    inputs = clip_processor(images=image, return_tensors="pt")
    
    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)
        
        # Safely extract PyTorch Tensor regardless of Transformers version
        if isinstance(outputs, torch.Tensor):
            features = outputs
        elif hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            features = outputs.pooler_output
        elif hasattr(outputs, "image_embeds") and outputs.image_embeds is not None:
            features = outputs.image_embeds
        else:
            features = outputs[0]

        # Normalize the embeddings to unit vectors
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().cpu().numpy()


def embed_text(text):
    """Embed Text using CLIP"""
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77  # CLIP's maximum token length for text
    )
    
    with torch.no_grad():
        outputs = clip_model.get_text_features(**inputs)
        
        # Safely extract PyTorch Tensor regardless of Transformers version
        if isinstance(outputs, torch.Tensor):
            features = outputs
        elif hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            features = outputs.pooler_output
        elif hasattr(outputs, "text_embeds") and outputs.text_embeds is not None:
            features = outputs.text_embeds
        else:
            features = outputs[0]

        # Normalize the embeddings to unit vectors
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().cpu().numpy()

In [31]:
# Process PDF and extract images and text
pdf_path = "multimodal_sample.pdf"
doc = fitz.open(pdf_path)

# Storage for all documents and embeddings
all_documents = []
all_embeddings = []
image_data_store = {}  # Store actual image data for later retrieval by LLM

# Text splitter for chunking text
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)



In [32]:
doc

Document('multimodal_sample.pdf')

In [12]:
# for i , page in enumerate(doc):
#     # Extract text
#     text = page.get_text()
#     if text.strip():  # Only process if there's text
#         text_chunks = splitter.split_text(text)
#         for chunk in text_chunks:
#             all_documents.append(Document(page_content=chunk, metadata={"source": f"Page {i+1}"}))
#             embedding = embed_text(chunk)
#             all_embeddings.append(embedding)

#     # Extract images
#     image_list = page.get_images(full=True)
#     for img_index, img in enumerate(image_list):
#         xref = img[0]
#         base_image = doc.extract_image(xref)
#         image_bytes = base_image["image"]
#         image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        
#         # Store the image data for later retrieval
#         image_key = f"Page_{i+1}_Image_{img_index+1}"
#         image_data_store[image_key] = image
        
#         # Embed the image
#         embedding = embed_image(image)
#         all_documents.append(Document(page_content=image_key, metadata={"source": f"Page {i+1}"}))
#         all_embeddings.append(embedding)

In [33]:
for i , page in enumerate(doc):
    # Process text
    text = page.get_text()
    if text.strip():
        # Only process if there's text

        # create temporary document for text chunking
        # temp_doc = Document(page_content=text, metadata={"source": f"Page {i+1}"})
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        # Embed each chunk using CLIP and store the embeddings
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_documents.append(chunk)
            all_embeddings.append(embedding)
        
    
    # Process images
    ##Three Important Actions:

    ##Convert PDF image to PIL format
    ##Store as base64 for GPT-4V (which needs base64 images)
    ##Create CLIP embedding for retrieval

    for img_index, img in enumerate(page.get_images(full=True)):
        try: 
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            # Convert to PIL image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
         
            # Store the image data as base64 for GPT-4V
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
            
            # Store the image data for later retrieval
            # Create unique identifier
            image_key = f"page_{i}_img_{img_index}"
            # image_key = f"Page_{i+1}_Image_{img_index+1}"
            image_data_store[image_key] = img_base64
            
            # Embed the image using CLIP
            embedding = embed_image(pil_image)
            # all_documents.append(Document(page_content=image_key, metadata={"page": i, "type": "image"}))
            all_embeddings.append(embedding)

            # Create document for the image 
            image_doc = Document(
                page_content=f"[Image: {image_key}]",
                metadata={"page": i, "type": "image" , "image_key": image_key}
                )
            all_documents.append(image_doc)

        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue


doc.close()
        
            
            


In [34]:
all_embeddings

[array([-2.67242640e-03,  1.28300013e-02, -5.18314503e-02,  4.14879806e-02,
        -2.33942028e-02, -7.55866850e-03, -3.67658995e-02,  1.19710714e-01,
         8.52081105e-02,  2.05426523e-03, -1.11534623e-02, -1.29592447e-02,
         5.25014624e-02, -3.65395704e-03,  4.76078652e-02,  1.58373136e-02,
         2.03388110e-02,  4.35361639e-02, -3.29167256e-03,  2.03181040e-02,
         1.88026344e-03, -4.23493907e-02,  5.44106448e-03,  3.70936021e-02,
        -1.65622756e-02,  6.48645451e-03, -4.78012338e-02,  8.67482647e-03,
         5.88860139e-02, -3.21394168e-02,  4.32440005e-02,  9.65301413e-03,
        -4.47922619e-03, -1.94857605e-02, -3.63502838e-02, -1.23471916e-02,
        -2.17929203e-02, -1.99016184e-02,  8.09620023e-02, -3.32986899e-02,
        -2.38901209e-02, -3.96139175e-02, -1.27280308e-02,  3.50380801e-02,
        -2.52216943e-02,  2.00031605e-03,  1.49660334e-02, -2.31976379e-02,
        -6.86791018e-02, -5.25785086e-04, -2.22545881e-02, -1.04103824e-02,
        -1.9

In [35]:
all_documents

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 0, 'type': 'image', 'image_key': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [36]:
# Create unified FAISS vector store with CLIP embeddings
embedding_array = np.array(all_embeddings)
embedding_array

array([[-0.00267243,  0.01283   , -0.05183145, ..., -0.00385087,
         0.02977719, -0.00010682],
       [ 0.01732336, -0.01327689, -0.0242703 , ...,  0.08994053,
        -0.00272156,  0.03253038]], shape=(2, 512), dtype=float32)

In [37]:
(all_documents, embedding_array)

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
  Document(metadata={'page': 0, 'type': 'image', 'image_key': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')],
 array([[-0.00267243,  0.01283   , -0.05183145, ..., -0.00385087,
          0.02977719, -0.00010682],
        [ 0.01732336, -0.01327689, -0.0242703 , ...,  0.08994053,
         -0.00272156,  0.03253038]], shape=(2, 512), dtype=float32))

In [40]:
# Create custome FAISS index since we have pre-computed embeddings
vector_store = FAISS.from_embeddings(
    text_embeddings= [(doc.page_content, emb) for doc, emb in zip(all_documents, embedding_array)],
    embedding=None, # as we have pre-computed embeddings, we don't need to pass an embedding function here
    metadatas = [doc.metadata for doc in all_documents]
    )

vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [41]:
# Initialize GPT-4 Vision model
llm = init_chat_model("openai:gpt-4.1")
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001BF8AE1BCB0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001BF8E2B0830>, root_client=<openai.OpenAI object at 0x000001BF8AE19160>, root_async_client=<openai.AsyncOpenAI object at 0x000001BF8E2B0590>, model_name='gpt-4.1', model_kwargs={}, openai_

In [42]:
def retrieve_multimodal(query, k=5):
    """Unified retrieval using CLIP embeddings for both text and images."""
    # Embed query using CLIP
    query_embedding = embed_text(query)
    
    # Search in unified vector store
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    
    return results

In [52]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for GPT-4V."""
    content = []
    
    # Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })
    
    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    
    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    
    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })
    
    return HumanMessage(content=content)

In [53]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)
    
    # Create multimodal message
    message = create_multimodal_message(query, context_docs)
    
    # Get response from GPT-4V
    response = llm.invoke([message])
    
    # Print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    return response.content

In [54]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 0: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
  - Image from page 0


Answer: Based on the provided context, the chart on page 1 shows that revenue increased steadily across Q1, Q2, and Q3. The chart illustrates:

- A moderate revenue increase in Q1 with the introduction of new product lines.
- A higher revenue growth in Q2, outperforming Q1, driven by marketing campaigns.
- The highest and exponential growth in Q3, attributed to global expansion.

Overall, the chart highlights a clear upward trend in revenue, with each quarter performing better than the last, and Q3 showing the most significant growth.

Query: Summarize the main findings from the document
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 0: Annual Revenue Ove

In [51]:
# def create_multimodal_message(query, retrieved_docs):
#     """Create a multimodal message for GPT-4 Vision."""
#     # Start with the query
#     messages = [HumanMessage(content=query)]
    
#     # Add retrieved documents
#     for doc in retrieved_docs:
#         if doc.metadata.get("type") == "image":
#             image_key = doc.metadata.get("image_key")
#             img_base64 = image_data_store.get(image_key)
#             if img_base64:
#                 # Include the base64 image in the message
#                 messages.append(HumanMessage(content=f"[Image: {image_key}]\n{img_base64}"))
#         else:
#             # For text, just include the content
#             messages.append(HumanMessage(content=doc.page_content))
    
#     return messages

In [ ]:
# def crete_multimodal_prompt(query, retrieved_docs):
#     """Create a prompt for GPT-4 Vision with retrieved multimodal content."""
#     # Start with the user query
#     prompt = f"User Query: {query}\n\n"
    
#     # Add retrieved documents
#     for i, doc in enumerate(retrieved_docs):
#         if doc.metadata.get("type") == "text":
#             prompt += f"Text Chunk {i+1} (Page {doc.metadata.get('page')}):\n{doc.page_content}\n\n"
#         elif doc.metadata.get("type") == "image":
#             image_key = doc.metadata.get("image_key")
#             img_base64 = image_data_store.get(image_key)
#             if img_base64:
#                 prompt += f"Image {i+1} (Page {doc.metadata.get('page')}): [Image: {image_key}]\nBase64: {img_base64}\n\n"
    
#     return prompt

In [ ]:
# def multimodal_query(query, k=5):
#     """Perform a multimodal query using GPT-4 Vision."""
#     # Retrieve relevant documents
#     retrieved_docs = retrieve_multimodal(query, k=k)
    
#     # Create a prompt for GPT-4 Vision
#     prompt = crete_multimodal_prompt(query, retrieved_docs)
    
#     # Get response from GPT-4 Vision
#     response = llm([HumanMessage(content=prompt)])
    
#     return response.content

In [ ]:
# def multimodal_rag(query, k=5):
#     """Perform a multimodal RAG query using GPT-4 Vision."""
#     # Retrieve relevant documents
#     retrieved_docs = retrieve_multimodal(query, k=k)
    
#     # Create a prompt for GPT-4 Vision
#     prompt = crete_multimodal_prompt(query, retrieved_docs)
    
#     # Get response from GPT-4 Vision
#     response = llm([HumanMessage(content=prompt)])
    
#     return response.content

In [49]:
# if __name__ == "main":
#     query = "Describe the image on page 2 and summarize the text on page 3."
#     response = multimodal_rag(query, k=5)
#     print("Response from GPT-4 Vision:")
#     print(response)